In [3]:
import re

# --- Part 1: basic replacement ---
text = "The course lasts 20 hours and contains 12 exercises."
print(re.sub(r"\d+", "<NUM>", text))
# The course lasts <NUM> hours and contains <NUM> exercises.

invoice = "The invoice #4521 totals $1,250.00, due in 30 days."
print(re.sub(r"\d+", "<NUM>", invoice))
# The invoice #<NUM> totals $<NUM>,<NUM>.<NUM>, due in <NUM> days.
# Observation: the invoice number, the due date and the amount are all
# replaced; the amount is fragmented into three tokens (1 / 250 / 00)
# because "," and "." are not matched by \d+.

# --- Part 2: pattern handling thousands separators (and decimals) ---
# \d{1,3}(?:,\d{3})+  -> numbers with comma thousands separators (1,250 / 1,250,000)
# \d+                 -> plain integers
# (?:\.\d+)?          -> optional decimal part (.00)
NUMBER_PATTERN = r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?"

print(re.sub(NUMBER_PATTERN, "<NUM>", invoice))
# The invoice #<NUM> totals $<NUM>, due in <NUM> days.

# --- Part 3: two Normalizer versions ---
class GenericNormalizer:
    """Replaces every number with a single placeholder <NUM>."""
    def __init__(self, pattern=NUMBER_PATTERN):
        self.pattern = re.compile(pattern)

    def normalize(self, text: str) -> str:
        return self.pattern.sub("<NUM>", text)


class MagnitudeNormalizer:
    """Replaces numbers with a placeholder keeping a coarse magnitude."""
    def __init__(self, pattern=NUMBER_PATTERN, threshold=100):
        self.pattern = re.compile(pattern)
        self.threshold = threshold

    def _replace(self, match: re.Match) -> str:
        value = float(match.group().replace(",", ""))
        return "<NUM_SMALL>" if value < self.threshold else "<NUM_LARGE>"

    def normalize(self, text: str) -> str:
        return self.pattern.sub(self._replace, text)


generic = GenericNormalizer()
magnitude = MagnitudeNormalizer()

print(generic.normalize(invoice))
# The invoice #<NUM> totals $<NUM>, due in <NUM> days.
print(magnitude.normalize(invoice))
# The invoice #<NUM_LARGE> totals $<NUM_LARGE>, due in <NUM_SMALL> days.

# --- Part 4: discussion (as comments) ---
# The magnitude version preserves more useful information for comparing
# invoice amounts: "<NUM_LARGE>" vs "<NUM_SMALL>" still tells the system
# that $1,250.00 is a big amount and 30 days is a small quantity, so
# amounts can be ranked at a coarse level (e.g. flag large invoices).
# The generic version collapses everything to <NUM>, making the amount,
# the invoice ID and the delay indistinguishable, so no comparison is
# possible. Limits of the magnitude version: with a single threshold of
# 100, $150 and $1,250,000 both become <NUM_LARGE>, and the invoice ID
# (#4521) is wrongly treated as a magnitude although it is an identifier,
# not a quantity. Finer buckets (e.g. <NUM_1E3>, <NUM_1E6>), or keeping
# the raw value in a separate field, would be needed for precise
# comparison; and identifiers should be handled by a separate pattern
# (e.g. r"#\d+" -> <ID>) before number normalization.

The course lasts <NUM> hours and contains <NUM> exercises.
The invoice #<NUM> totals $<NUM>,<NUM>.<NUM>, due in <NUM> days.
The invoice #<NUM> totals $<NUM>, due in <NUM> days.
The invoice #<NUM> totals $<NUM>, due in <NUM> days.
The invoice #<NUM_LARGE> totals $<NUM_LARGE>, due in <NUM_SMALL> days.


In [2]:
import re
from nltk.tokenize import word_tokenize
import nltk

# nltk.download("punkt", quiet=True)
# nltk.download("punkt_tab", quiet=True)

comment = ("OMG this courseeee is SOOOO good!!! 10/10 would recommend :) "
           "Visit https://univ.dz/nlp for more info. Contact: prof@univ.dz")

# --- Part 1: cleaning pipeline ---
URL_PATTERN = r"https?://\S+|www\.\S+"
EMAIL_PATTERN = r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"

def clean_text(text: str) -> str:
    # 1. Remove URLs and emails FIRST, on the original text: they are
    #    case-insensitive anyway, and later steps (repeated-letter collapse,
    #    whitespace) must not corrupt them (e.g. "www" or "aaa" in a URL).
    text = re.sub(URL_PATTERN, "<URL>", text)
    text = re.sub(EMAIL_PATTERN, "<EMAIL>", text)
    # 2. Lowercase (after placeholders are inserted, so they would be
    #    lowercased too: re-insert them in lowercase form for consistency)
    text = text.lower()
    # 3. Collapse repeated letters: coursEEEE -> coursee, soooo -> soo
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    # 4. Normalize whitespace (tabs/newlines/multiple spaces -> single space)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Part 2: before / after side by side ---
cleaned = clean_text(comment)
print(f"{'BEFORE':<10}: {comment}")
print(f"{'AFTER':<10}: {cleaned}")
# BEFORE    : OMG this courseeee is SOOOO good!!! 10/10 would recommend :) Visit https://univ.dz/nlp for more info. Contact: prof@univ.dz
# AFTER     : omg this coursee is soo good!! 10/10 would recommend :) visit <url> for more info. contact: <email>

# --- Part 3: vocabulary size before / after ---
tokens_before = word_tokenize(comment)
tokens_after = word_tokenize(cleaned)

print("\nTokens before:", tokens_before)
print("Tokens after :", tokens_after)
print(f"\nVocabulary size before: {len(set(tokens_before))}")
print(f"Vocabulary size after : {len(set(tokens_after))}")
print(f"Total tokens before   : {len(tokens_before)}")
print(f"Total tokens after    : {len(tokens_after)}")
# Typical result: the cleaned version has fewer total tokens (the URL and
# email, which word_tokenize splits into many fragments such as 'https',
# ':', '//univ.dz/nlp', 'prof', '@', 'univ.dz', collapse into <url>/<email>)
# and merges case variants (OMG/omg, SOOOO/soo), so the vocabulary shrinks
# and the remaining types are more reusable across comments.
# Note: word_tokenize splits "<url>" into '<', 'url', '>' (3 tokens).
# Using a placeholder without angle brackets (e.g. "URL_TOKEN") or a
# custom tokenizer avoids this.

# --- Part 4: discussion (as comments) ---
# For sentiment analysis, aggressively removing exclamation marks and
# repeated letters is a BAD idea. Both are emotional intensity signals:
#  - "soooo good" and "so good" differ in strength; elongation marks
#    emphasis and enthusiasm (or sarcasm), and "!!!" signals excitement.
#  - Removing them erases information that the classifier could use to
#    distinguish "good" from "VERY good", or "awful" from "AWFUL!!!".
# What the pipeline above does is a reasonable compromise: it COLLAPSES
# repetitions to at most two (rather than deleting them), so elongated
# words like "soooo" map to a single shared form "soo" (reducing
# vocabulary sparsity) while a trace of the emphasis survives, and it
# keeps punctuation. Even better for sentiment: keep "!!!" (or map it to
# a feature/token such as <EXCL>), keep emoticons like ":)" (strong
# polarity cues), and preserve ALL-CAPS as a flag (<CAPS>) before
# lowercasing. URLs and emails, in contrast, carry no sentiment and are
# safe to replace or remove.

BEFORE    : OMG this courseeee is SOOOO good!!! 10/10 would recommend :) Visit https://univ.dz/nlp for more info. Contact: prof@univ.dz
AFTER     : omg this coursee is soo good!! 10/10 would recommend :) visit <url> for more info. contact: <email>

Tokens before: ['OMG', 'this', 'courseeee', 'is', 'SOOOO', 'good', '!', '!', '!', '10/10', 'would', 'recommend', ':', ')', 'Visit', 'https', ':', '//univ.dz/nlp', 'for', 'more', 'info', '.', 'Contact', ':', 'prof', '@', 'univ.dz']
Tokens after : ['omg', 'this', 'coursee', 'is', 'soo', 'good', '!', '!', '10/10', 'would', 'recommend', ':', ')', 'visit', '<', 'url', '>', 'for', 'more', 'info', '.', 'contact', ':', '<', 'email', '>']

Vocabulary size before: 23
Vocabulary size after : 22
Total tokens before   : 27
Total tokens after    : 26


# Exercise 5: Normalization Decisions Across Applications

| Text element | Application | Decision | Short justification |
|---|---|---|---|
| Capital letters | Sentiment analysis | **Preserve** (or encode as a flag) | ALL-CAPS signals intensity: "TERRIBLE" is stronger than "terrible". |
| Numbers | Financial document analysis | **Preserve** | Amounts, dates and rates are the core content; "$1,250" vs "$125,000" changes everything. |
| Punctuation | Question answering | **Preserve** | "?" marks the question type, and commas/quotes/periods affect parsing and answer boundaries. |
| Multiple spaces / line breaks | Web-scraped corpus cleaning | **Normalize** | They are layout artifacts (HTML indentation, templates), so collapse them to a single space. |
| Emojis | Social-media sentiment analysis | **Preserve** (or map to a sentiment token) | Emojis are often the strongest polarity cue: "great 🙄" is sarcastic, "great 😍" is not. |
| Repeated letters ("soooo good") | Social-media sentiment analysis | **Normalize** (collapse, don't delete) | Collapsing to "soo" reduces vocabulary sparsity while keeping a trace of emphasis. |
| Accented characters (naïve, café) | Multilingual search engine | **Normalize** (strip accents for matching) | Users often type "cafe" for "café", so matching should be accent-insensitive. |

---

## Justifications and counter-situations

### 1. Capital letters (sentiment analysis): preserve
- **Why:** In "The service was AWFUL", the capitals carry emotional intensity. Lowercasing loses that signal.
- **Opposite case:** For topic classification or search over formal documents, lowercasing is the right choice. "Apple" and "apple" at the start of a sentence should count as the same word, and it reduces vocabulary size.

### 2. Numbers (financial document analysis): preserve
- **Why:** Replacing "$1,250.00" with `<NUM>` makes an invoice indistinguishable from a $5 receipt, so amounts can no longer be compared or validated.
- **Opposite case:** For document-type classification (invoice vs. contract vs. letter), exact values do not matter, and normalizing to `<NUM>` helps the model generalize across documents.

### 3. Punctuation (question answering): preserve
- **Why:** "Let's eat, grandma" vs. "Let's eat grandma" shows that punctuation can change meaning, and "?" helps identify the question type.
- **Opposite case:** For bag-of-words topic modeling or keyword extraction, punctuation is noise and is usually removed.

### 4. Multiple spaces / line breaks (web-scraped corpus): normalize
- **Why:** `"Price:\n\n     50   DA"` should become `"Price: 50 DA"`, since the extra whitespace is an HTML artifact and not content.
- **Opposite case:** For poetry, code, or tables, line breaks and indentation are structural, and collapsing them destroys the meaning (e.g., Python indentation, verse lines).

### 5. Emojis (social-media sentiment): preserve
- **Why:** 😡 and 😂 are strong polarity cues that often matter more than the words around them.
- **Opposite case:** For a formal-document pipeline, or a model whose tokenizer cannot handle emojis (producing `<UNK>`), removing them is the safer choice.

### 6. Repeated letters (social-media sentiment): normalize
- **Why:** "goooood", "gooood" and "good" are the same word, so mapping them to one form helps the classifier. Collapsing to at most two letters keeps a mark of emphasis, while full deletion would lose it.
- **Opposite case:** If the task is to detect emotional intensity or sarcasm, elongation itself is the feature and should be kept (or converted into an explicit `<ELONGATED>` flag).

### 7. Accented characters (multilingual search engine): normalize
- **Why:** A search for "cafe" should find "café", so accent stripping (e.g., Unicode NFD decomposition, then removing combining marks) improves recall.
- **Opposite case:** In French or Arabic-transliteration contexts where accents distinguish words ("a" = has vs. "à" = to; "ou" = or vs. "où" = where), or for spell-checking and machine translation, accents must be preserved.

---

## Key takeaway

There is no universally correct preprocessing. Each normalization step trades **vocabulary reduction** (less sparsity, better generalization) against **information loss** (signal the task may need). The decision should always depend on what the downstream task is trying to detect.